# 02 · Limpeza dos dados

Aplica à base bruta as correções **determinísticas** (que não dependem de estatísticas dos dados) e salva a base limpa em `data/processed/sleep_clean.csv`.

A validação não encontrou problemas de qualidade. Mesmo assim, a limpeza é executada por completo: ela confirma que a base continua íntegra e deixa o processo pronto para uma nova versão dos dados.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from scroll_sono.cleaning import clean
from scroll_sono.config import CLEAN_DATA_PATH, FEATURE_SETS, ID_COL, TARGET
from scroll_sono.data import load_clean, load_raw, save_clean

In [2]:
raw = load_raw()
print(f"Base bruta: {raw.shape[0]:,} linhas × {raw.shape[1]} colunas")

Base bruta: 8,500 linhas × 18 colunas


## 1. Correções

As funções ficam em `scroll_sono.cleaning` e são aplicadas nesta ordem:

| Passo | O que faz |
|---|---|
| Padronizar texto | Remove espaços nas pontas e espaços repetidos das colunas categóricas, do alvo e do identificador |
| Remover duplicatas | Remove linhas repetidas, inclusive as que só diferem pelo `user_id` |
| Converter para número | Garante que as colunas numéricas sejam numéricas; valores não numéricos viram `NaN` |
| Valores fora do domínio → `NaN` | Valores impossíveis (ex.: horas negativas) viram `NaN`, para serem imputados no pré-processamento |

A imputação de valores ausentes não é feita aqui, porque depende de estatísticas (média, moda) que devem ser calculadas apenas no conjunto de treino.

In [3]:
df, relatorio = clean(raw)
relatorio

,alterações,linhas após o passo
passo,,
padronizar texto,0,8500
remover duplicatas,0,8500
converter para número,0,8500
valores fora do domínio → NaN,0,8500


## 2. Verificação da base limpa

In [4]:
verificacoes = pd.Series({
    "linhas": len(df),
    "colunas": df.shape[1],
    "valores ausentes": int(df.isna().sum().sum()),
    "linhas duplicadas": int(df.duplicated().sum()),
    "tipos iguais aos da base bruta": bool((df.dtypes == raw.dtypes).all()),
    "base idêntica à bruta": df.equals(raw),
})
verificacoes.to_frame("resultado")

,resultado
linhas,8500
colunas,18
valores ausentes,0
linhas duplicadas,0
tipos iguais aos da base bruta,True
base idêntica à bruta,True


**Resultado:** nenhum passo alterou a base. Ela continua com 8.500 linhas e 18 colunas, sem valores ausentes nem duplicatas, e é idêntica à base bruta.

## 3. Colunas mantidas

Todas as colunas são mantidas na base limpa, inclusive as que não entram nos modelos:

- `user_id` permite rastrear cada registro até a base original.
- As colunas medidas durante o sono e no dia seguinte são usadas na análise exploratória.

A escolha dos preditores é feita na modelagem, pelos cenários de atributos definidos em `scroll_sono.config.FEATURE_SETS`:

In [5]:
pd.DataFrame({
    "cenário": list(FEATURE_SETS),
    "atributos": [len(cols) for cols in FEATURE_SETS.values()],
    "colunas fora do cenário": [
        ", ".join(c for c in df.columns if c not in cols and c not in (ID_COL, TARGET))
        or "—"
        for cols in FEATURE_SETS.values()
    ],
}).set_index("cenário")

,atributos,colunas fora do cenário
cenário,,
comportamental,10,"sleep_latency_min, total_sleep_hours, deep_sle..."
completo,16,—


## 4. Salvar a base limpa

In [6]:
save_clean(df)
print("Salvo em:", CLEAN_DATA_PATH.relative_to(CLEAN_DATA_PATH.parents[2]))

Salvo em: data/processed/sleep_clean.csv


In [7]:
# A base salva é lida de volta igual à que foi gerada?
recarregada = load_clean()
pd.testing.assert_frame_equal(recarregada, df)
print("Leitura conferida:", recarregada.shape)

Leitura conferida: (8500, 18)


## Resumo

| Passo | Alterações |
|---|---|
| Padronizar texto | 0 |
| Remover duplicatas | 0 |
| Converter para número | 0 |
| Valores fora do domínio → `NaN` | 0 |

A base limpa, `data/processed/sleep_clean.csv`, tem as mesmas 8.500 linhas e 18 colunas da base bruta. Ela é a entrada da etapa 3 (separação treino/teste).